# Whisper small: baseline vs. fine-tuned, re-scored with the shared evaluation code

Team Tokens · COE025 Project 3 · Finnish ASR

Runs both Whisper systems on the three test sets with `scripts/asr_eval.py`, the evaluation code that
every system in this project uses (same clips, same references, one `normalize()`, corpus-level WER/CER).

| System | Checkpoint | Role |
|---|---|---|
| whisper-small zero-shot | `openai/whisper-small` | baseline (no training by us) |
| whisper-small fine-tuned (ours) | `tia88na/whisper-small-fi-tokens` | contribution (trained by us, `scripts/finnish_asr_whisper_finetune.ipynb`) |

No training happens here. Both systems use the `openai/whisper-small` processor (the one used for fine-tuning).
Decoding is the same as in the fine-tuning notebook: language = Finnish,
task = transcribe, greedy, max 225 new tokens, fp16. The earlier numbers are kept unchanged in `results/legacy/`.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all (≈ 40–60 min).


**Reproduction run (2026-11-22).** The committed results in `results/scores/` and `results/predictions/`
stay the reference. This run writes to `results_rerun/` (not committed) and the last cell compares the
two runs, so the saved outputs below show that the evaluation runs end to end and reproduces the
committed numbers. After Run all: **File → Download → Download .ipynb**.

In [1]:
!pip -q install -U transformers datasets jiwer accelerate soundfile librosa

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.2 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
# Get the repo (shared evaluation code + clip lists) and work inside it
import os, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"   # plain HTTP download; the Xet downloader can hang on Colab
if not os.path.isdir("/content/team-tokens-finnish-asr"):
    !git clone -q https://github.com/zehraoz36/team-tokens-finnish-asr /content/team-tokens-finnish-asr
else:
    !git -C /content/team-tokens-finnish-asr pull -q
%cd /content/team-tokens-finnish-asr
sys.path.insert(0, "scripts")
import asr_eval as E
print(E.hardware())
import torch
if not torch.cuda.is_available():
    raise SystemExit("NO GPU: Colab gave a CPU runtime (GPU quota used up?). "
                     "Runtime -> Change runtime type -> T4 GPU, or try later / another account.")

/content/team-tokens-finnish-asr
Tesla T4 (torch 2.14.1+cu130)


In [3]:
# Test sets: test_cv, test_parl, test_fleurs (16 kHz). References = `text` column.
test = E.load_test_sets()
for s, d in test.items():
    print(f"{s:12s} {len(d):5d} clips")

README.md:   0%|          | 0.00/988 [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/270M [00:00<?, ?B/s]

dev_cv-00000-of-00001.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

dev_parl-00000-of-00002.parquet:   0%|          | 0.00/306M [00:00<?, ?B/s]

dev_parl-00001-of-00002.parquet:   0%|          | 0.00/289M [00:00<?, ?B/s]

... (7 outputs left out)

Generating test_cv split:   0%|          | 0/1833 [00:00<?, ? examples/s]

Generating test_fleurs split:   0%|          | 0/916 [00:00<?, ? examples/s]

Generating test_parl split:   0%|          | 0/1913 [00:00<?, ? examples/s]

test_cv       1833 clips
test_parl     1913 clips
test_fleurs    916 clips


In [4]:
# Reproduction run: write to results_rerun/ so the committed results stay untouched
OUT_SCORES, OUT_PRED = "results_rerun/scores", "results_rerun/predictions"

## Transcription function

`transcribe_fn(list of 16 kHz float arrays) -> list of raw strings`. The shared code times only this call (RTF).

In [5]:
import torch, transformers
from transformers import WhisperForConditionalGeneration, WhisperProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
GEN = dict(language="finnish", task="transcribe", num_beams=1, do_sample=False, max_new_tokens=225)
SETTINGS = (f"language=fi, task=transcribe, greedy, max_new_tokens=225, 30 s window, "
            f"transformers {transformers.__version__}")

# Both systems use the processor (log-mel front end + tokenizer) of openai/whisper-small:
# the fine-tuning notebook trained with exactly this processor, and the fine-tuned repo
# stores it only in the newer single-file format (processor_config.json).
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

def make_whisper(model_id):
    model = WhisperForConditionalGeneration.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE).eval()
    model.generation_config.forced_decoder_ids = None

    @torch.no_grad()
    def transcribe(arrays):
        feats = processor.feature_extractor(arrays, sampling_rate=16000, return_tensors="pt").input_features
        ids = model.generate(feats.to(DEVICE, DTYPE), **GEN)
        return processor.batch_decode(ids, skip_special_tokens=True)
    return model, transcribe

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

## Baseline: `openai/whisper-small` zero-shot

In [6]:
BASELINE = "whisper-small zero-shot"
model, fn = make_whisper("openai/whisper-small")
E.evaluate_system(BASELINE, "openai/whisper-small", fn, test, role="baseline",
                  settings=SETTINGS, precision=str(DTYPE).split(".")[-1], batch_size=16,
                  scores_dir=OUT_SCORES, pred_dir=OUT_PRED)
del model, fn; torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

whisper-small zero-shot test_cv:   0%|          | 0/115 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers a

[whisper-small zero-shot] test_cv      WER= 30.94  CER=  6.71  RTF=0.0202


... (2 outputs left out)

[whisper-small zero-shot] test_parl    WER= 25.68  CER=  8.40  RTF=0.0152


whisper-small zero-shot test_fleurs:   0%|          | 0/58 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[whisper-small zero-shot] test_fleurs  WER= 26.37  CER=  5.26  RTF=0.0114


## Contribution: our fine-tuned model `tia88na/whisper-small-fi-tokens`

In [7]:
FINETUNED = "whisper-small fine-tuned (ours)"
model, fn = make_whisper("tia88na/whisper-small-fi-tokens")
E.evaluate_system(FINETUNED, "tia88na/whisper-small-fi-tokens", fn, test, role="contribution",
                  settings=SETTINGS + "; trained on train split, best dev-WER checkpoint",
                  precision=str(DTYPE).split(".")[-1], batch_size=16,
                  scores_dir=OUT_SCORES, pred_dir=OUT_PRED)
del model, fn; torch.cuda.empty_cache()

config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/484M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/4.58k [00:00<?, ?B/s]

whisper-small fine-tuned (ours) test_cv:   0%|          | 0/115 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

... (3 outputs left out)

[whisper-small fine-tuned (ours)] test_parl    WER= 24.72  CER=  5.89  RTF=0.0144


whisper-small fine-tuned (ours) test_fleurs:   0%|          | 0/58 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[whisper-small fine-tuned (ours)] test_fleurs  WER= 28.85  CER=  5.97  RTF=0.0110


## Reproduction check: this run vs. the committed results

In [8]:
display(E.compare_runs([BASELINE, FINETUNED], OUT_SCORES, OUT_PRED))

,system,test_set,wer_committed,wer_rerun,cer_committed,cer_rerun,identical_outputs
0,whisper-small zero-shot,test_cv,30.94,30.94,6.71,6.71,1833/1833
1,whisper-small zero-shot,test_parl,25.68,25.68,8.40,8.40,1913/1913
2,whisper-small zero-shot,test_fleurs,26.37,26.37,5.26,5.26,916/916
3,whisper-small fine-tuned (ours),test_cv,23.93,23.93,4.61,4.61,1833/1833
4,whisper-small fine-tuned (ours),test_parl,24.72,24.72,5.89,5.89,1913/1913
5,whisper-small fine-tuned (ours),test_fleurs,28.85,28.85,5.97,5.97,916/916


Done. Save this executed notebook: **File → Download → Download .ipynb**.